# Dynamic compaction

`CompactionSettings` controls a closed-loop series of transactional cell
changes. Each trial changes the cell, relaxes, checks guards, and is
accepted or rolled back before the next increment.

In [ ]:
# Compaction is configured independently from the recipe using it.
import tangle
from tangle.units import mm, um

## Every `CompactionSettings` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `target` | Stopping observable, as a target object (see the next table). | `VolumeFractionTarget`, `CellVolumeTarget`, `CellLengthsTarget`, `MeanPressureTarget`, `DirectionalPressureTarget`, `PenaltyEnergyTarget` |
| `path` | Rule for distributing cell motion among axes (see below). | `AxisWeightsPath`, `EqualPressurePath`, `StressRatioPath`, `MinimumWorkPath` |
| `kinematics` | How geometry follows cell changes. | `rigid_fiber_centers`, `moving_walls`, `affine_vertices` |
| `cell_anchor` | Stationary fractional point while each cell axis changes. | xyz in [0,1] |
| `balance_opposing_faces` | Balances work between low and high faces. | boolean |
| `face_pressure_floor` | Floor used by opposing-face balancing. | pressure |
| `face_balance_strength` | Strength of opposing-face feedback. | 0–1 |
| `initial_log_strain` | First attempted logarithmic strain increment. | positive strain |
| `min_log_strain` | Smallest retry increment. | positive strain |
| `max_log_strain` | Largest grown increment. | positive strain |
| `growth_factor` | Increment multiplier after easy accepted steps. | greater than 1 |
| `shrink_factor` | Increment multiplier after rejected steps. | between 0 and 1 |
| `relax_iterations` | Relaxation work allotted to each increment window. | count |
| `max_shortening_over_min_diameter` | Caps an increment by the thinnest fiber size. | ratio |
| `max_penetration` | Rejects a trial exceeding this overlap. | m |
| `max_curvature_ratio` | Rejects a trial exceeding this curvature utilization. | ratio |
| `max_pressure` | Pressure guard. | pressure |
| `max_penalty_energy` | Formation-energy guard. | energy |
| `max_steps` | Maximum accepted/retried compaction steps. | count |
| `max_relax_windows` | Maximum windows spent settling one trial. | count |
| `contact_energy_stiffness` | Contact contribution to the formation penalty. | model stiffness |
| `stretch_energy_stiffness` | Stretch contribution to the formation penalty. | model stiffness |
| `bending_energy_stiffness` | Bending contribution to the formation penalty. | model stiffness |
| `target_tolerance` | Relative/absolute acceptance tolerance for the target. | target-dependent |

In [ ]:
# Read defaults from the compiled extension instead of duplicating
# them in documentation that could become stale.
compaction = tangle.CompactionSettings()
fields = ['target', 'path', 'kinematics', 'cell_anchor', 'balance_opposing_faces', 'face_pressure_floor', 'face_balance_strength', 'initial_log_strain', 'min_log_strain', 'max_log_strain', 'growth_factor', 'shrink_factor', 'relax_iterations', 'max_shortening_over_min_diameter', 'max_penetration', 'max_curvature_ratio', 'max_pressure', 'max_penalty_energy', 'max_steps', 'max_relax_windows', 'contact_energy_stiffness', 'stretch_energy_stiffness', 'bending_energy_stiffness', 'target_tolerance']
{name: getattr(compaction, name) for name in fields}

## Targets and paths

The target says when to stop; the path says how the cell moves to get
there. Both are small objects that carry only their own options, and
they are chosen independently.

| Target | Stops at | Value |
| --- | --- | --- |
| `VolumeFractionTarget(value)` | Stop at a nominal fiber volume fraction. | 0–1 |
| `CellVolumeTarget(value)` | Stop at a cell volume. | m³ |
| `CellLengthsTarget(lengths)` | Stop at three cell edge lengths. | xyz, m |
| `MeanPressureTarget(value)` | Stop at a mean wall pressure. | pressure |
| `DirectionalPressureTarget(pressures)` | Stop at per-axis wall pressures. | xyz pressure |
| `PenaltyEnergyTarget(value)` | Stop at a formation-penalty energy. | energy |

| Path | Meaning | Options |
| --- | --- | --- |
| `AxisWeightsPath(weights=None)` | Prescribed relative shortening; `None` shortens only the stack axis. | `"z"`, `"xy"`, or nonnegative xyz weights |
| `EqualPressurePath(axes, pressure_floor=...)` | Feedback that equalizes pressure on the active axes. | axis set such as `"xyz"` |
| `StressRatioPath(ratio, pressure_floor=...)` | Feedback toward a directional pressure ratio. | nonnegative xyz |
| `MinimumWorkPath(axes)` | Moves whichever active axis needs the least incremental work. | axis set such as `"xy"` |

## Common target/path combinations

`CompactionSettings.volume_fraction(v, **changes)` is the concise
constructor: it sets `VolumeFractionTarget(v)` and the default
`AxisWeightsPath()`, which shortens only the recipe's stack axis. Any
other field follows as a keyword. `replace()` then swaps a target or
path while keeping every guard.

In [ ]:
# Only the stack axis (z for a periodic="xy" cell) shortens by default.
compaction = tangle.CompactionSettings.volume_fraction(
    0.40,
    kinematics="moving_walls",
    cell_anchor=[0.5, 0.5, 0.5],  # both z faces move
    balance_opposing_faces=True,
    max_penetration=0.1 * um,
    max_curvature_ratio=1.05,
)

# replace() compares paths and targets without rebuilding every guard.
explicit_z = compaction.replace(path=tangle.AxisWeightsPath("z"))
equal_pressure = compaction.replace(path=tangle.EqualPressurePath("xyz"))
stress_ratio = compaction.replace(
    path=tangle.StressRatioPath([1.0, 1.0, 2.0], pressure_floor=1.0)
)
least_work = compaction.replace(path=tangle.MinimumWorkPath("xy"))
target_lengths = compaction.replace(
    target=tangle.CellLengthsTarget([0.8 * mm, 0.8 * mm, 1.2 * mm])
)
target_pressure = compaction.replace(target=tangle.MeanPressureTarget(1.0e3))

# The constructor takes the target object directly, too.
by_volume = tangle.CompactionSettings(
    tangle.CellVolumeTarget(0.8e-9), path=tangle.AxisWeightsPath([1.0, 1.0, 1.0])
)
by_direction = tangle.CompactionSettings(tangle.DirectionalPressureTarget([0.0, 0.0, 1.0e3]))
by_energy = tangle.CompactionSettings(tangle.PenaltyEnergyTarget(1.0e-9))
[compaction.target, compaction.path, equal_pressure.path, target_lengths.target]

In [ ]:
# Overrides affect only the compaction operation they are passed to.
recipe = tangle.Recipe(tangle.Cell([1 * mm, 1 * mm, 2 * mm], periodic="xy"))
# A gentle first densification with a relaxation preset...
recipe.compact(
    tangle.CompactionSettings.volume_fraction(0.13, kinematics="moving_walls"),
    tangle.RelaxationOverrides.preset("contact_first"),
)
# ...then the guarded final compaction with a hand-written override.
recipe.compact(compaction, tangle.RelaxationOverrides(contact_aggregation="deepest_only"))
recipe.operations()